In [ ]:
import model_inference as mi
from importlib import reload
import pandas as pd
from data_extraction import main as extract_data
from feature_derivation import main as derive_features

#data_folder = extract_data()

data_folder = "Data_16_06_2026"


In [ ]:
# X_train, y_train, X_inference = derive_features(
#     data_folder=data_folder
# )

X_train.to_parquet(f'{data_folder}/preprocessed_data/X_train.parquet')
y_train.to_frame().to_parquet(f'{data_folder}/preprocessed_data/y_train.parquet')
X_inference.to_parquet(f'{data_folder}/preprocessed_data/X_inference.parquet')

X_train = pd.read_parquet(f'{data_folder}/preprocessed_data/X_train.parquet')
y_train = pd.read_parquet(f'{data_folder}/preprocessed_data/y_train.parquet').close_log_return
X_inference = pd.read_parquet(f'{data_folder}/preprocessed_data/X_inference.parquet')

In [ ]:
params = {'num_parallel_tree': 10, "colsample_bytree": 0.8, "colsample_bynode": 0.8}

preds, metrics = mi.back_test(
    X_train,
    y_train,
    whole_back_test=True,
    **params
)

  0%|          | 0/16 [00:00<?, ?it/s][11:54:07] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1744329155408/work/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

 25%|██▌       | 4/16 [21:32<1:04:38, 323.21s/it] 


In [46]:
preds[preds.marketcap >= preds.marketcap_quantile].groupby('calendardate').tail(20).groupby('calendardate').y_true.mean()

calendardate
2010-03-31    0.928787
2010-06-30    1.056975
2010-09-30    0.315050
2010-12-31    0.143283
2011-03-31    0.220577
                ...   
2024-03-31    1.582990
2024-06-30    2.611912
2024-09-30    4.572766
2024-12-31    1.794924
2025-03-31    3.332789
Name: y_true, Length: 61, dtype: float32

In [98]:
preds[
    (preds.close >= 0.25*preds.close_max)
    & (preds.marketcap >= preds.marketcap_quantile)
].groupby('calendardate').tail(10).groupby('calendardate').y_true.mean()

calendardate
2010-03-31    0.955902
2010-06-30    1.418755
2010-09-30    0.389720
2010-12-31    0.054463
2011-03-31    0.317391
                ...   
2024-03-31    0.977508
2024-06-30    1.215046
2024-09-30    2.669239
2024-12-31    2.326719
2025-03-31    5.084293
Name: y_true, Length: 61, dtype: float32

In [58]:
preds[
    (preds.close >= 0.25*preds.close_max)
    & (preds.marketcap >= preds.marketcap_quantile)
].groupby('calendardate').tail(10).groupby('calendardate').y_true.mean().describe()

count    61.000000
mean      1.027672
std       1.130344
min      -0.106938
25%       0.361362
50%       0.752531
75%       1.152175
max       6.597761
Name: y_true, dtype: float64

In [8]:
params = {'num_parallel_tree': 10, "colsample_bytree": 0.8, "colsample_bynode": 0.8}

inference_predictions = mi.fit_and_predict(
    X_train,
    y_train,
    X_inference,
    **params
)

In [9]:
reload(mi)
inference_performance_to_date = mi.obtain_inference_performance_to_date(
    inference_predictions,
    marketcap_quantile=0.25,
    data_folder=data_folder
)

In [6]:
inference_predictions.to_parquet(f'{data_folder}/results/inference_predictions.parquet')

accoci        assets  assetsavg       assetsc  \
ticker calendardate                                                      
A      2000-03-31    53000000.0  7.107000e+09        NaN  4.982000e+09   
       2000-06-30    -4000000.0  7.321000e+09        NaN  5.057000e+09   
       2000-09-30    -6000000.0  7.827000e+09        NaN  5.344000e+09   
       2001-03-31    33000000.0  9.208001e+09        NaN  5.461000e+09   
       2001-06-30    15000000.0  9.080001e+09        NaN  4.998000e+09   
...                         ...           ...        ...           ...   
ZZ     2010-09-30    -2644000.0  9.648830e+08        NaN  3.681870e+08   
       2011-03-31    10733000.0  9.490990e+08        NaN  3.576150e+08   
       2011-06-30    11824000.0  9.326520e+08        NaN  3.430990e+08   
       2011-09-30    10703000.0  9.478520e+08        NaN  3.631280e+08   
       2012-03-31     5986000.0  9.362590e+08        NaN  3.599860e+08   

                         assetsnc  assetturnover    bvps        capex  \
ticker calendardate                                                     
A      2000-03-31    2.125000e+09            NaN  10.219  -30000000.0   
       2000-06-30    2.264000e+09            NaN  10.270  -95000000.0   
       2000-09-30    2.483000e+09            NaN  10.821 -222000000.0   
       2001-03-31    3.747000e+09            NaN  12.178 -114000000.0   
       2001-06-30    4.082000e+09            NaN  12.316    5000000.0   
...                           ...            ...     ...          ...   
ZZ     2010-09-30    5.966960e+08            NaN  -0.983   -4691000.0   
       2011-03-31    5.914840e+08            NaN  -0.758   -5703000.0   
       2011-06-30    5.895530e+08            NaN  -0.720   -7514000.0   
       2011-09-30    5.847240e+08            NaN  -0.569   -4451000.0   
       2012-03-31    5.762730e+08            NaN  -0.627   -1841000.0   

                          cashneq    cashnequsd  ...  Utilities_shareswadil  \
ticker calendardate                              ...                          
A      2000-03-31    1.368000e+09  1.368000e+09  ...             47398516.0   
       2000-06-30    9.780000e+08  9.780000e+08  ...             52512000.0   
       2000-09-30    7.030000e+08  7.030000e+08  ...             54200000.0   
       2001-03-31    4.330000e+08  4.330000e+08  ...             61694000.0   
       2001-06-30    8.090000e+08  8.090000e+08  ...             55900000.0   
...                           ...           ...  ...                    ...   
ZZ     2010-09-30    8.251000e+07  8.251000e+07  ...             77850000.0   
       2011-03-31    1.023900e+08  1.023900e+08  ...             79950000.0   
       2011-06-30    7.929200e+07  7.929200e+07  ...             78600000.0   
       2011-09-30    8.593400e+07  8.593400e+07  ...             79100000.0   
       2012-03-31    9.792400e+07  9.792400e+07  ...             80750000.0   

                     Utilities_sps  Utilities_tangibles  Utilities_taxassets  \
ticker calendardate                                                            
A      2000-03-31           5.7760         1.847346e+09             135656.0   
       2000-06-30           6.0150         1.923533e+09                  0.0   
       2000-09-30           5.0200         2.100710e+09                  0.0   
       2001-03-31           7.9625         2.711210e+09                  0.0   
       2001-06-30           7.8530         2.604950e+09                  0.0   
...                            ...                  ...                  ...   
ZZ     2010-09-30           4.6600         3.761998e+09            3868000.0   
       2011-03-31           5.0095         4.496500e+09           13510000.0   
       2011-06-30           5.6850         4.071225e+09           10028000.0   
       2011-09-30           4.9350         4.168626e+09            8237500.0   
       2012-03-31           5.2890         4.477812e+09            7800000.0   

                     Utilities_taxexp  Util

In [114]:
inference_performance_to_date[
    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)
    & (inference_performance_to_date.calendardate == '2026-03-31')
    #& (inference_performance_to_date.ticker != 'SHAZ')
].sort_values('pct_change').groupby('calendardate').tail(10).pct_change_at_max_date.describe()

count    10.000000
mean      0.559574
std       0.610396
min      -0.048649
25%       0.328892
50%       0.373774
75%       0.550908
max       2.088869
Name: pct_change_at_max_date, dtype: float64

In [ ]:
inference_performance_to_date[    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)].sort_values('pct_change').groupby('calendardate').tail(10).sort_values('calendardate')#.groupby('calendardate').pct_change_at_max_date.describe().T

,ticker,calendardate,pct_change,pct_change_at_max_date,close,close_max,marketcap,marketcap_quantile
11057,SION,2025-06-30,1.262805,1.074928,17.350000,25.000000,5.692047e+08,86157792.0
13871,ZBIO,2025-06-30,1.118549,0.930857,9.690000,25.680000,4.074649e+08,86157792.0
10839,SEPN,2025-06-30,0.735976,2.408704,10.570000,27.090000,4.509304e+08,86157792.0
11234,SLSR,2025-06-30,0.645662,1.179039,4.580000,4.680000,6.311112e+08,86157792.0
10139,RHLD,2025-06-30,0.916303,3.132413,31.870001,50.490002,2.397511e+08,86157792.0
842,ABVX,2025-06-30,0.730902,11.844444,7.650000,15.890000,4.027091e+08,86157792.0
7741,LION,2025-06-30,0.746286,1.473322,5.810000,8.150000,2.065744e+09,86157792.0
6204,HUHU,2025-06-30,0.684534,0.652263,4.860000,5.960000,1.112663e+08,86157792.0
11356,SNDK,2025-06-30,1.220922,45.479824,45.349998,56.419998,5.785815e+09,86157792.0
9905,RAPP,2025-06-30,0.691502,2.364996,11.370000,29.230000,3.886990e+08,86157792.0


In [23]:
inference_performance_to_date[
    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)
    & (inference_performance_to_date.calendardate == '2026-03-31')
].sort_values('pct_change').groupby('calendardate').tail(20).pct_change_at_max_date.describe()

count    20.000000
mean      0.605425
std       0.787699
min      -0.221198
25%       0.069317
50%       0.373774
75%       0.836248
max       2.558725
Name: pct_change_at_max_date, dtype: float64

In [18]:
selected_stocks = [
    "NTSK",
    "INV",
    "MNTN",
    "CHA",
    "AGBK",
    "KLAR",
    "SSII",
    "BETA",
    "WOLF",
    "FLY",
    "EQPT",
    "WYFI",
    "OMDA",
    "TLX",
    "IBTA",
    "FIGR",
    "BBNX",
    "MANE",
    "LMRI",
    "PICS",
    "SAIL",
    "ANTA",
    "XZO",
    "ETOR",
    "PTRN",
    "AERO",
    "HTFL",
    "GLXY",
    "WLTH",
    "BLLN"
]

len(selected_stocks)

30

In [27]:
inference_performance_to_date[
    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)
].sort_values('pct_change').groupby('calendardate').tail(10).groupby('calendardate').pct_change_at_max_date.describe().T

calendardate,2025-06-30,2025-09-30,2025-12-31,2026-03-31
count,10.000000,10.000000,10.000000,10.000000
mean,7.054079,2.918786,0.539036,0.881145
std,13.897722,5.388759,0.670275,0.950976
min,0.652263,0.224073,-0.477842,-0.048649
25%,1.100956,0.587371,0.233048,0.351463
50%,1.919159,0.831389,0.520542,0.424225
75%,2.951486,1.754483,0.927518,1.650182
max,45.479824,17.786631,1.819644,2.558725


In [ ]:
inference_performance_to_date[
    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)
].groupby('calendardate').pct_change_at_max_date.describe().T

calendardate,2025-06-30,2025-09-30,2025-12-31,2026-03-31
count,1929.000000,2012.000000,1990.000000,1993.000000
mean,0.289621,0.157561,0.131190,0.126753
std,1.332310,0.691076,0.484974,0.324752
min,-0.999740,-0.979647,-0.976624,-0.808616
25%,-0.120637,-0.139042,-0.104169,-0.023889
50%,0.117495,0.061708,0.058145,0.060178
75%,0.408447,0.303114,0.237168,0.197085
max,45.479824,17.786631,7.879687,2.628977


In [20]:
inference_performance_to_date[
    inference_performance_to_date.ticker.isin(selected_stocks)
].sort_values('pct_change').groupby('calendardate').pct_change_at_max_date.describe().T

calendardate,2025-06-30,2025-09-30,2025-12-31,2026-03-31
count,7.000000,10.000000,17.000000,23.000000
mean,-0.215562,-0.073330,0.183064,0.295690
std,0.264048,0.218675,0.600128,0.512175
min,-0.441457,-0.472131,-0.528060,-0.149273
25%,-0.404705,-0.216430,-0.290157,0.016136
50%,-0.378390,-0.014045,0.131512,0.133333
75%,-0.065881,0.087066,0.437799,0.346745
max,0.252083,0.157271,1.819644,2.007966


In [22]:
inference_performance_to_date[
    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)
].sort_values('pct_change').groupby('calendardate').tail(20).groupby('calendardate').pct_change_at_max_date.mean()

calendardate
2025-06-30    2.654480
2025-09-30    0.551877
2025-12-31    0.049873
2026-03-31    0.000000
Name: pct_change_at_max_date, dtype: float64

In [5]:
inference_performance_to_date[
    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)
].sort_values('pct_change').groupby('calendardate').tail(20).groupby('calendardate').pct_change_at_max_date.mean()

calendardate
2025-03-31    2.134912
2025-06-30    2.092430
2025-09-30    0.568499
2025-12-31    0.217706
Name: pct_change_at_max_date, dtype: float64